<a href="https://colab.research.google.com/github/lonaliakbar/lonaliakbar.github.io/blob/main/SPJ_2025_FIXED_v9_NAMA_KAPITAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pymupdf pypdf pdfplumber

from google.colab import auth
auth.authenticate_user()
import google.auth
creds, _ = google.auth.default()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
import io, re, fitz, pdfplumber, random
from datetime import datetime
from pypdf import PdfReader, PdfWriter

drive_service = build('drive', 'v3', credentials=creds)

# --- Konfigurasi ---
SOURCE_ROOT_ID = '14NSO7VzkoGZJgXJycyE378_GyS42lGmF'
OUTPUT_ROOT_NAME_BASE = 'SPJ 2025 FIXED'

# 1. SPP-2 SEKARANG MASUK KE DALAM SERI EDIT TANGGAL SAMA SEPERTI SPP-1, SPP-3, DAN SPTB
EDIT_PATTERN = re.compile(r'^(SPP-1|SPP-2|SPP-3|SPTB)\.pdf$', re.IGNORECASE)
SPP1_PATTERN = re.compile(r'^SPP-1\.pdf$', re.IGNORECASE)
SPTB_PATTERN = re.compile(r'^SPTB\.pdf$', re.IGNORECASE)

# [PERBAIKAN REGEX] - Menghapus double backslash pada raw string
KWT_PATTERN = re.compile(r'^KWT-\d+-DARI-\d+\.pdf$', re.IGNORECASE)
FOLDER_NUMBER_PATTERN = re.compile(r'^(\d+)\s*-\s*.*$')

SHIFT_POINTS = 20
GABUNG_NAME = "SPP-12-Gabung.pdf"

# 2. VARIABEL CONFIG JAM RANDOM (Bisa diatur rentang jam di sini)
JAM_MODE = "random"  # Set ke "random" untuk mengaktifkan jam acak otomatis

MONTHS = {
    'january':1, 'february':2, 'march':3, 'april':4, 'may':5, 'june':6,
    'july':7, 'august':8, 'september':9, 'october':10, 'november':11, 'december':12,
    'januari':1, 'februari':2, 'maret':3, 'mei':5, 'juni':6,
    'juli':7, 'agustus':8, 'oktober':10, 'desember':12,
}

log_lines = []
def log(msg):
    print(msg)
    log_lines.append(msg)

# --- Drive helper ---
def list_folder_items(folder_id):
    items, page_token = [], None
    while True:
        resp = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token, supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute()
        items.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return items

def download_file(file_id):
    request = drive_service.files().get_media(fileId=file_id)
    buf = io.BytesIO()
    downloader = MediaIoBaseDownload(buf, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buf.seek(0)
    return buf

def upload_file(name, data_bytes, parent_id):
    media = MediaIoBaseUpload(io.BytesIO(data_bytes), mimetype='application/pdf',
                              resumable=True, chunksize=1024*1024)
    meta = {'name': name, 'parents': [parent_id]}
    request = drive_service.files().create(body=meta, media_body=media, fields='id', supportsAllDrives=True)
    response = None
    while response is None:
        status, response = request.next_chunk()
    return response

def create_folder(name, parent_id):
    meta = {'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]}
    return drive_service.files().create(body=meta, fields='id', supportsAllDrives=True).execute()['id']

def find_child_folder_by_name(parent_id, name):
    resp = drive_service.files().list(
        q=f"'{parent_id}' in parents and name = '{name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
        fields="files(id, name)", supportsAllDrives=True, includeItemsFromAllDrives=True
    ).execute()
    files = resp.get('files', [])
    return files[0]['id'] if files else None

# --- Fungsi cari teks berbasis posisi (baca PDF) ---
def find_line_above(page, label_text):
    rects = page.search_for(label_text)
    if not rects:
        return None
    label_rect = rects[0]
    d = page.get_text("dict")
    best, best_gap = None, float("inf")
    for b in d["blocks"]:
        for line in b.get("lines", []):
            text = "".join(s["text"] for s in line["spans"]).strip()
            if not text or text == label_text.strip():
                continue
            bbox = fitz.Rect(line["bbox"])
            if bbox.y1 <= label_rect.y0 + 2 and abs(bbox.x0 - label_rect.x0) < 15:
                gap = label_rect.y0 - bbox.y1
                if 0 <= gap < 40 and gap < best_gap:
                    best_gap = gap
                    best = {"text": text, "bbox": bbox, "size": line["spans"][0]["size"]}
    return best

def find_line_right_of(page, label_text):
    rects = page.search_for(label_text)
    if not rects:
        return None
    label_rect = rects[0]
    d = page.get_text("dict")
    best, best_gap = None, float("inf")
    for b in d["blocks"]:
        for line in b.get("lines", []):
            text = "".join(s["text"] for s in line["spans"]).strip()
            if not text or text == label_text.strip():
                continue
            bbox = fitz.Rect(line["bbox"])
            if bbox.x0 >= label_rect.x1 - 2 and abs(bbox.y0 - label_rect.y0) < 5:
                gap = bbox.x0 - label_rect.x1
                if 0 <= gap < 60 and gap < best_gap:
                    best_gap = gap
                    best = {"text": text, "bbox": bbox, "size": line["spans"][0]["size"]}
    return best

def get_field_value(page, label_text):
    rects = page.search_for(label_text)
    if not rects:
        return None
    label_rect = rects[0]
    d = page.get_text("dict")
    candidates = []
    for b in d["blocks"]:
        for line in b.get("lines", []):
            text = "".join(s["text"] for s in line["spans"]).strip()
            if not text or text == label_text.strip():
                continue
            bbox = fitz.Rect(line["bbox"])
            if abs(bbox.y0 - label_rect.y0) < 3 and bbox.x0 > label_rect.x1 - 5:
                candidates.append((bbox.x0, text))
    if not candidates:
        return None
    candidates.sort(key=lambda c: c[0])
    return candidates[-1][1]

def find_exact_lines(page, target_text):
    d = page.get_text("dict")
    found = []
    for b in d["blocks"]:
        for line in b.get("lines", []):
            text = "".join(s["text"] for s in line["spans"]).strip()
            if text == target_text:
                found.append({"bbox": fitz.Rect(line["bbox"]), "size": line["spans"][0]["size"]})
    return found

def get_reference_date(doc):
    for page in doc:
        kd = find_line_above(page, "Pelaksana Kegiatan,")
        if kd:
            m = re.search(r'(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})', kd["text"])
            if m:
                day, month_name, year = m.group(1), m.group(2).lower(), m.group(3)
                month = MONTHS.get(month_name)
                if month:
                    return f"{int(day):02d}/{month:02d}/{year}"
    return None

def generate_random_time():
    """Fungsi pembantu untuk membuat jam random antara jam 00:00:00 - 06:59:59"""
    h = random.randint(0, 6)
    m = random.randint(0, 59)
    s = random.randint(0, 59)
    return f" {h:02d}:{m:02d}:{s:02d}"

def apply_reference_date(doc, reference_date):
    changed = False
    for page in doc:
        pd = find_line_right_of(page, "Printed by Siskeudes")
        if not pd:
            continue
        m2 = re.match(r'(\d{2}/\d{2}/\d{4})(\s+\d{2}:\d{2}:\d{2})', pd["text"])
        if not m2:
            continue

        if JAM_MODE == "random":
            time_part = generate_random_time()
        else:
            time_part = m2.group(2)

        new_full = reference_date + time_part
        if new_full == pd["text"]:
            continue
        bbox = pd["bbox"]
        page.add_redact_annot(bbox, fill=(1, 1, 1))
        page.apply_redactions()
        page.insert_text((bbox.x0, bbox.y1 - 2), new_full, fontsize=pd["size"], fontname="helv")
        changed = True
    return changed

def transform_name_uppercase(name):
    words = name.split()
    return " ".join(w if w.lower() == "dkk" else w.upper() for w in words)

def get_sptb_penerima_names(pdf_bytes):
    names = set()
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    if not row or len(row) < 2 or not row[1]:
                        continue
                    if not re.match(r'^\d+$', str(row[0] or '').strip()):
                        continue
                    name = str(row[1]).split("\n")[0].strip()
                    if name:
                        names.add(name)
    return names

def apply_sptb_name_uppercase(doc, names):
    changed = False
    for name in names:
        new_name = transform_name_uppercase(name)
        if new_name == name:
            continue
        for page in doc:
            for occ in find_exact_lines(page, name):
                bbox = occ["bbox"]
                page.add_redact_annot(bbox, fill=(1, 1, 1))
                page.apply_redactions()
                page.insert_text((bbox.x0, bbox.y1 - 2), new_name, fontsize=occ["size"], fontname="helv")
                changed = True
    return changed

# --- [FIX] Kapitalisasi nama pada blok tanda tangan (Pelaksana Kegiatan / Disetujui / Dibayar / Verifikasi / Menerima / Memberi) ---
SIGNATURE_LABELS = [
    "Pelaksana Kegiatan,",
    "Disetujui untuk dibayarkan",
    "Telah dibayar lunas",
    "Telah dilakukan verifikasi",
    "Yang Menerima,",
    "Yang Memberi,",
]

def find_bottom_line_below(page, label_text, max_gap=150):
    """Cari baris PALING BAWAH yang sejajar kolom di bawah label.
    Pada blok tanda tangan Siskeudes urutannya: Label -> Jabatan -> (spasi ttd) -> Nama,
    jadi nama selalu berada paling bawah sebelum baris 'Printed by Siskeudes'."""
    rects = page.search_for(label_text)
    if not rects:
        return None
    label_rect = rects[0]
    footer_rects = page.search_for("Printed by Siskeudes")
    footer_y = footer_rects[0].y0 if footer_rects else label_rect.y1 + max_gap
    d = page.get_text("dict")
    best = None
    for b in d["blocks"]:
        for line in b.get("lines", []):
            text = "".join(s["text"] for s in line["spans"]).strip()
            if not text or text == label_text.strip():
                continue
            bbox = fitz.Rect(line["bbox"])
            if bbox.y0 > label_rect.y1 and bbox.y0 < footer_y and abs(bbox.x0 - label_rect.x0) < 15:
                if best is None or bbox.y0 > best["bbox"].y0:
                    best = {"text": text, "bbox": bbox, "size": line["spans"][0]["size"]}
    return best

def apply_signature_names_uppercase(doc):
    """Kapitalisasi nama di semua blok tanda tangan (SPP-1, SPP-2, SPP-3, SPTB, KWT)."""
    changed = False
    for page in doc:
        for label in SIGNATURE_LABELS:
            nd = find_bottom_line_below(page, label)
            if not nd:
                continue
            name = nd["text"]
            new_name = transform_name_uppercase(name)
            if new_name == name:
                continue
            bbox = nd["bbox"]
            page.add_redact_annot(bbox, fill=(1, 1, 1))
            page.apply_redactions()
            page.insert_text((bbox.x0, bbox.y1 - 2), new_name, fontsize=nd["size"], fontname="helv")
            changed = True
    return changed

def shift_margin_top(pdf_bytes, shift_points=SHIFT_POINTS):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    writer = PdfWriter()
    for page in reader.pages:
        page.add_transformation((1, 0, 0, 1, 0, shift_points))
        writer.add_page(page)
    out = io.BytesIO()
    writer.write(out)
    return out.getvalue()

def merge_pdfs(bytes_list):
    writer = PdfWriter()
    for b in bytes_list:
        reader = PdfReader(io.BytesIO(b))
        for page in reader.pages:
            writer.add_page(page)
    out = io.BytesIO()
    writer.write(out)
    return out.getvalue()

def make_unique_root_folder(base_name):
    name = base_name
    counter = 1
    while True:
        resp = drive_service.files().list(
            q=f"'root' in parents and name = '{name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
            fields="files(id, name)"
        ).execute()
        if not resp.get('files'):
            folder_id = create_folder(name, 'root')
            log(f"Folder root dibuat: '{name}'")
            return folder_id, name
        name = f"{base_name} - {counter}"
        counter += 1

def get_source_subfolders(root_id):
    items = list_folder_items(root_id)
    return [(sf['id'], sf['name'], list_folder_items(sf['id']))
            for sf in items if sf['mimeType'] == 'application/vnd.google-apps.folder']

# === EKSEKUSI ===
OUTPUT_ROOT_ID, OUTPUT_ROOT_NAME = make_unique_root_folder(OUTPUT_ROOT_NAME_BASE)

log("Mengambil subfolder di dalam folder sumber ...")
folders_found = get_source_subfolders(SOURCE_ROOT_ID)
log(f"Ditemukan {len(folders_found)} subfolder.\n")

total_folder, total_file_ok, total_file_gagal = 0, 0, 0

for idx, (src_folder_id, folder_name, items) in enumerate(folders_found, start=1):
    log(f"=== Folder sumber: {folder_name} ===")

    spp1_candidates = [f for f in items if SPP1_PATTERN.match(f['name'])]
    if not spp1_candidates:
        log(f"  -> SPP-1.pdf tidak ditemukan, folder dilewati\n")
        continue
    spp1_file = spp1_candidates[0]

    m = FOLDER_NUMBER_PATTERN.match(folder_name)
    number = m.group(1) if m else str(idx)

    spp1_buf = download_file(spp1_file['id'])
    spp1_doc = fitz.open(stream=spp1_buf.read(), filetype="pdf")
    keperluan = None
    for page in spp1_doc:
        keperluan = get_field_value(page, "Keperluan")
        if keperluan:
            break
    reference_date = get_reference_date(spp1_doc)
    spp1_doc.close()

    if not keperluan:
        keperluan = folder_name
        log(f"  -> Keperluan tidak ditemukan, pakai nama folder asli sebagai fallback")
    if not reference_date:
        log(f"  -> Tanggal acuan tidak ditemukan di SPP-1, folder ini dilewati\n")
        continue

    new_subfolder_name = f"{number} - {keperluan}"
    log(f"  -> Nama subfolder baru: '{new_subfolder_name}'")
    log(f"  -> Tanggal acuan: {reference_date}")

    new_subfolder_id = find_child_folder_by_name(OUTPUT_ROOT_ID, new_subfolder_name)
    if not new_subfolder_id:
        new_subfolder_id = create_folder(new_subfolder_name, OUTPUT_ROOT_ID)
        log(f"  -> Subfolder dibuat di '{OUTPUT_ROOT_NAME}'")
    total_folder += 1

    processed_bytes = {}

    # 1. Proses SPP-1, SPP-2, SPP-3, SPTB (SEMUA SINKRON TANGGAL & JAM RANDOM)
    edit_files = [f for f in items if EDIT_PATTERN.match(f['name'])]
    for f in edit_files:
        try:
            raw_bytes = download_file(f['id']).read()
            doc = fitz.open(stream=raw_bytes, filetype="pdf")
            apply_reference_date(doc, reference_date)

            # [FIX] Kapitalisasi nama pada blok tanda tangan — berlaku untuk SPP-1, SPP-2, SPP-3, SPTB
            if apply_signature_names_uppercase(doc):
                log(f"    Nama pada blok tanda tangan diubah huruf besar (kecuali 'dkk')")

            if SPTB_PATTERN.match(f['name']):
                names = get_sptb_penerima_names(raw_bytes)
                if apply_sptb_name_uppercase(doc, names):
                    log(f"    Nama Penerima (tabel) diubah huruf besar (kecuali 'dkk')")

            out_buf = io.BytesIO()
            doc.save(out_buf)
            doc.close()
            final_bytes = out_buf.getvalue()
            processed_bytes[f['name']] = final_bytes

            upload_file(f['name'], final_bytes, new_subfolder_id)
            log(f"  -> Diproses & diupload: {f['name']}")
            total_file_ok += 1
        except Exception as e:
            log(f"  -> GAGAL: {f['name']} — {e}")
            total_file_gagal += 1

    # 2. Proses file KWT: sinkron tanggal + geser margin atas
    kwt_files = sorted([f for f in items if KWT_PATTERN.match(f['name'])], key=lambda f: f['name'])
    for f in kwt_files:
        try:
            raw_bytes = download_file(f['id']).read()
            doc = fitz.open(stream=raw_bytes, filetype="pdf")
            apply_reference_date(doc, reference_date)

            # [FIX] Kapitalisasi nama pada blok tanda tangan (Yang Menerima / Yang Memberi) di file KWT
            if apply_signature_names_uppercase(doc):
                log(f"    Nama pada blok tanda tangan KWT diubah huruf besar (kecuali 'dkk')")

            mid_buf = io.BytesIO()
            doc.save(mid_buf)
            doc.close()

            final_bytes = shift_margin_top(mid_buf.getvalue())
            processed_bytes[f['name']] = final_bytes

            upload_file(f['name'], final_bytes, new_subfolder_id)
            log(f"  -> Diproses (tanggal+margin) & diupload: {f['name']}")
            total_file_ok += 1
        except Exception as e:
            log(f"  -> GAGAL: {f['name']} — {e}")
            total_file_gagal += 1

    # 3. Gabungkan Semua File Secara Berurutan (Menggunakan teks asli SPP-2 hasil edit)
    try:
        gabung_order = []
        spp1_bytes = next((v for k, v in processed_bytes.items() if k.lower() == "spp-1.pdf"), None)
        if spp1_bytes:
            gabung_order.append(spp1_bytes)

        # SPP-2 SEKARANG DIAMBIL DARI HASIL PROSES EDIT TEKS ASLI (BUKAN FOTO RASTER LAGI)
        spp2_bytes = next((v for k, v in processed_bytes.items() if k.lower() == "spp-2.pdf"), None)
        if spp2_bytes:
            gabung_order.append(spp2_bytes)

        sptb_bytes = next((v for k, v in processed_bytes.items() if k.lower() == "sptb.pdf"), None)
        if sptb_bytes:
            gabung_order.append(sptb_bytes)
        for f in kwt_files:
            if f['name'] in processed_bytes:
                gabung_order.append(processed_bytes[f['name']])

        if gabung_order:
            gabung_bytes = merge_pdfs(gabung_order)
            upload_file(GABUNG_NAME, gabung_bytes, new_subfolder_id)
            log(f"  -> Digabung & diupload: {GABUNG_NAME} ({len(gabung_order)} dokumen)")
            total_file_ok += 1
        else:
            log(f"  -> Tidak ada file untuk digabung")
    except Exception as e:
        log(f"  -> GAGAL menggabung: {e}")
        total_file_gagal += 1

    log("")

log(f"Selesai! {total_folder} subfolder diproses, {total_file_ok} file berhasil, {total_file_gagal} file gagal.")

# --- Simpan log ke file txt (lokal Colab + upload ke folder output) ---
log_filename = f"log_proses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(log_filename, "w", encoding="utf-8") as fp:
    fp.write("\n".join(log_lines))

with open(log_filename, "rb") as fp:
    upload_file(log_filename, fp.read(), OUTPUT_ROOT_ID)

print(f"\nLog tersimpan: {log_filename} (lokal Colab & di folder '{OUTPUT_ROOT_NAME}')")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 55.4 MB/s eta 0:00:00


Folder root dibuat: 'SPJ 2025 FIXED'
Mengambil subfolder di dalam folder sumber ...
Ditemukan 109 subfolder.

=== Folder sumber: 8 - Pembayaran Siltap Perangkat ===
  -> Nama subfolder baru: '8 - Pembayaran Siltap Perangkat'
  -> Tanggal acuan: 06/05/2025
  -> Subfolder dibuat di 'SPJ 2025 FIXED'
    Nama pada blok tanda tangan diubah huruf besar (kecuali 'dkk')
  -> Diproses & diupload: SPP-1.pdf
    Nama pada blok tanda tangan diubah huruf besar (kecuali 'dkk')
    Nama Penerima (tabel) diubah huruf besar (kecuali 'dkk')
  -> Diproses & diupload: SPTB.pdf
    Nama pada blok tanda tangan diubah huruf besar (kecuali 'dkk')
  -> Diproses & diupload: SPP-2.pdf
    Nama pada blok tanda tangan KWT diubah huruf besar (kecuali 'dkk')
  -> Diproses (tanggal+margin) & diupload: KWT-1-DARI-3.pdf
    Nama pada blok tanda tangan KWT diubah huruf besar (kecuali 'dkk')
  -> Diproses (tanggal+margin) & diupload: KWT-2-DARI-3.pdf
    Nama pada blok tanda tangan KWT diubah huruf besar (kecuali 'dkk')


KeyboardInterrupt: 

In [ ]:
# --- CELL BARU 1: KUMPULKAN KEPERLUAN & BUAT FILE EXCEL ---
import io
import fitz
import pandas as pd
from googleapiclient.http import MediaIoBaseUpload

print(f"Mengumpulkan data keperluan dari folder sumber ke folder hasil: {OUTPUT_ROOT_NAME}")
folders_found = get_source_subfolders(SOURCE_ROOT_ID)

excel_data = []

for idx, (src_folder_id, folder_name, items) in enumerate(folders_found, start=1):
    spp1_candidates = [f for f in items if SPP1_PATTERN.match(f['name'])]
    if not spp1_candidates:
        continue
    spp1_file = spp1_candidates[0]

    # Ambil nomor folder
    m = FOLDER_NUMBER_PATTERN.match(folder_name)
    number = m.group(1) if m else str(idx)

    # Ambil teks Keperluan dari SPP-1
    try:
        spp1_buf = download_file(spp1_file['id'])
        spp1_buf.seek(0)
        spp1_doc = fitz.open(stream=spp1_buf.read(), filetype="pdf")
        keperluan = None
        for page in spp1_doc:
            keperluan = get_field_value(page, "Keperluan")
            if keperluan:
                break
        spp1_doc.close()
    except Exception as e:
        keperluan = None

    if not keperluan:
        keperluan = folder_name  # Fallback jika tidak ketemu

    excel_data.append({"nomor": str(number), "keperluan": keperluan})
    print(f" Terdata -> Nomor: {number} | Keperluan: {keperluan}")

# Buat DataFrame Pandas dan konversi ke Excel (.xlsx) di dalam memori
df = pd.DataFrame(excel_data)
excel_buffer = io.BytesIO()
with pd.ExcelWriter(excel_buffer, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Keperluan')

# Upload file Excel ke Google Drive di dalam folder FIXED-nya
excel_bytes = excel_buffer.getvalue()
media = MediaIoBaseUpload(io.BytesIO(excel_bytes), mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', resumable=True)
meta = {'name': 'daftar_keperluan.xlsx', 'parents': [OUTPUT_ROOT_ID]}

try:
    # Cek apakah file sudah ada sebelumnya agar tidak duplikat (opsi overwrite)
    query = f"'{OUTPUT_ROOT_ID}' in parents and name = 'daftar_keperluan.xlsx' and trashed = false"
    existing_files = drive_service.files().list(q=query, fields="files(id)", supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get('files', [])

    if existing_files:
        file_id = existing_files[0]['id']
        excel_file_drive = drive_service.files().update(fileId=file_id, media_body=media, supportsAllDrives=True).execute()
        print(f"\n[SUKSES] File 'daftar_keperluan.xlsx' berhasil diperbarui di Drive!")
    else:
        excel_file_drive = drive_service.files().create(body=meta, media_body=media, fields='id', supportsAllDrives=True).execute()
        print(f"\n[SUKSES] File 'daftar_keperluan.xlsx' berhasil dibuat di Drive!")

    print(f"ID File Excel: {excel_file_drive['id']}")
except Exception as e:
    print(f"\n[GAGAL] Gagal menyimpan file Excel ke Drive: {e}")

In [ ]:
# --- CELL BARU 2: CEK SPREADSHEET & EDIT PDF JIKA BEDA (FULL & DEBUG VER.) ---
import io
import fitz
import pandas as pd
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

print("Mencari dan membaca file 'daftar_keperluan.xlsx' dari Drive...\n")

# 1. Cari dan download Excel terbaru dari Drive
query = f"'{OUTPUT_ROOT_ID}' in parents and name = 'daftar_keperluan.xlsx' and trashed = false"
existing_files = drive_service.files().list(q=query, fields="files(id)", supportsAllDrives=True).execute().get('files', [])

if not existing_files:
    print("File daftar_keperluan.xlsx tidak ditemukan di Drive.")
else:
    file_id = existing_files[0]['id']
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()

    fh.seek(0)
    df = pd.read_excel(fh, sheet_name='Keperluan')
    print("Berhasil membaca data dari spreadsheet. Memulai pengecekan...\n")
    print("-" * 50)

    # Ambil ulang data folder sumber
    folders_found = get_source_subfolders(SOURCE_ROOT_ID)

    # 2. Mulai proses pengecekan baris per baris
    for idx, row in df.iterrows():
        # CLEANING: Kadang pandas baca angka jadi float (misal '1.0'), kita hapus desimalnya dan spasinya
        nomor_raw = str(row['nomor']).strip()
        if nomor_raw.endswith('.0'):
            nomor_raw = nomor_raw[:-2]
        nomor_excel = nomor_raw

        keperluan_excel = str(row['keperluan']).strip()

        print(f"[CEK] Baris Excel -> Nomor: '{nomor_excel}' | Keperluan Target: '{keperluan_excel}'")

        # Cari folder yang sesuai dengan nomor
        target_folder = None
        for src_id, f_name, f_items in folders_found:
            m = FOLDER_NUMBER_PATTERN.match(f_name)
            # Samakan perlakuan cleaning ke nomor folder di Drive
            f_num = str(m.group(1)).strip() if m else str(idx + 1)

            if f_num == nomor_excel:
                target_folder = (src_id, f_name, f_items)
                print(f"      Ketemu folder Drive: {f_name}")
                break

        if not target_folder:
            print(f"      [!] SKIP: Folder untuk nomor '{nomor_excel}' tidak ditemukan di Drive.")
            print("-" * 50)
            continue

        _, _, folder_items = target_folder

        # Ambil file SPP-1 master & Gabungan
        spp1_files = [f for f in folder_items if SPP1_PATTERN.match(f['name'])]
        gabungan_files = [f for f in folder_items if "gabung" in f['name'].lower() or "merged" in f['name'].lower()]
        files_to_check = spp1_files + gabungan_files

        if not files_to_check:
            print(f"      [!] SKIP: Tidak ada file SPP-1/Gabungan di folder ini.")
            print("-" * 50)
            continue

        # 3. Cek masing-masing PDF
        for pdf_file in files_to_check:
            pdf_id = pdf_file['id']
            pdf_name = pdf_file['name']

            try:
                pdf_buf = download_file(pdf_id)
                pdf_buf.seek(0)
                doc = fitz.open(stream=pdf_buf.read(), filetype="pdf")

                butuh_update = False

                for page in doc:
                    # Ambil teks lama di PDF dan bersihkan spasinya
                    keperluan_pdf_raw = get_field_value(page, "Keperluan")
                    keperluan_pdf = str(keperluan_pdf_raw).strip() if keperluan_pdf_raw else ""

                    # Jika teks beda, eksekusi proses edit (cover + tulis ulang)
                    if keperluan_pdf and keperluan_pdf != keperluan_excel:
                        print(f"      -> [BEDA] Perbedaan ditemukan pada: {pdf_name}")
                        print(f"         Teks Asli PDF : '{keperluan_pdf}'")
                        print(f"         Teks Di Excel : '{keperluan_excel}'")
                        print(f"         Melakukan proses tip-ex (cover) & edit...")

                        text_instances = page.search_for("Keperluan")
                        for rect in text_instances:
                            # Buat kotak putih penutup di sebelah kanan teks "Keperluan"
                            cover_rect = fitz.Rect(rect.x1 + 10, rect.y0 - 2, rect.x1 + 300, rect.y1 + 2)
                            page.draw_rect(cover_rect, color=(1, 1, 1), fill=(1, 1, 1))

                            # Tulis nilai baru
                            page.insert_text(fitz.Point(rect.x1 + 12, rect.y1 - 2), keperluan_excel, fontsize=10, color=(0, 0, 0))

                        butuh_update = True
                    elif keperluan_pdf == keperluan_excel:
                        print(f"      -> [SAMA] File {pdf_name} sudah sesuai, tidak ada perubahan.")

                # 4. Jika ada editan, upload & overwrite file di Drive
                if butuh_update:
                    out_buf = io.BytesIO()
                    doc.save(out_buf, garbage=4, deflate=True)
                    out_buf.seek(0)

                    media = MediaIoBaseUpload(out_buf, mimetype='application/pdf', resumable=True)
                    drive_service.files().update(fileId=pdf_id, media_body=media, supportsAllDrives=True).execute()
                    print(f"      -> [SUKSES] File {pdf_name} berhasil diperbarui di Drive.")

                doc.close()

            except Exception as e:
                print(f"      -> [ERROR] Gagal memproses {pdf_name}: {e}")

        print("-" * 50)

print("\nSelesai mengecek dan memperbarui PDF.")

In [ ]:
try:
    import reportlab
except ModuleNotFoundError:
    print("[PROSES] Menginstal library 'reportlab' ke Google Colab...")
    import os
    os.system('pip install reportlab')

try:
    import fitz  # PyMuPDF
except ModuleNotFoundError:
    print("[PROSES] Menginstal library 'PyMuPDF' ke Google Colab...")
    import os
    os.system('pip install pymupdf')

import io
import fitz
import pandas as pd
import re
import threading
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.graphics.shapes import Drawing, Rect, String
from reportlab.graphics.barcode import qr

# --- PASTIKAN VARIABEL BERIKUT SUDAH ADA SEBELUMNYA ---
# Jika belum ada, silakan uncomment dan sesuaikan baris di bawah ini:
# OUTPUT_ROOT_ID = "ID_FOLDER_DRIVE_ANDA"
# GABUNG_NAME = "Gabungan_LPJ.pdf"
# FOLDER_NUMBER_PATTERN sudah didefinisikan di Cell 0 (format: "1 - Nama Keperluan")
# KWT_PATTERN = re.compile(r"^kwt.*", re.IGNORECASE)

print("Memulai pembuatan Cover Premium dengan Standar Layout Eksekutif Clean...")

# Inisialisasi map agar tidak menyebabkan NameError
if 'updated_keperluan_map' not in locals():
    updated_keperluan_map = {}

# 1. Sinkronisasi database dari daftar_keperluan.xlsx
if not updated_keperluan_map:
    print("[PROSES] Membaca database 'daftar_keperluan.xlsx' dari Drive...")
    query = f"'{OUTPUT_ROOT_ID}' in parents and name = 'daftar_keperluan.xlsx' and trashed = false"
    try:
        request = drive_service.files().list(q=query, fields="files(id)", supportsAllDrives=True, includeItemsFromAllDrives=True)
        excel_search = request.execute().get('files', [])
    except Exception as err:
        print(f"[API ERROR] Gagal list file excel: {err}")
        excel_search = []

    if excel_search:
        try:
            excel_buf = download_file(excel_search[0]['id'])
            excel_buf.seek(0)
            df_updated = pd.read_excel(excel_buf, engine='openpyxl')
            for _, row in df_updated.iterrows():
                if 'nomor' in df_updated.columns and 'keperluan' in df_updated.columns:
                    val_nomor = row['nomor']
                    if pd.isna(val_nomor): continue
                    str_nomor = str(int(val_nomor)) if isinstance(val_nomor, float) and val_nomor.is_integer() else str(val_nomor).strip()
                    updated_keperluan_map[str_nomor] = str(row['keperluan']).strip()
        except Exception as ex:
            print(f"[ERROR] Gagal memproses file Excel: {ex}")

if updated_keperluan_map:
    output_folders = list_folder_items(OUTPUT_ROOT_ID)
    total_cover_created = 0

    # 2. MODIFIKASI INPUT: Mendukung angka, koma, rentang, dan TIMEOUT 2 MENIT
    tes_input = ""

    def minta_input():
        global tes_input
        tes_input = input("Masukkan nomor folder (Misal: 1, 1-9, 1,3,4). Waktu Anda 2 menit, atau kosongkan untuk SEMUA: ").strip()

    # Menjalankan input di thread terpisah agar bisa diberi batas waktu
    t = threading.Thread(target=minta_input)
    t.start()

    # Tunggu maksimal 120 detik (2 menit)
    t.join(timeout=120.0)

    if t.is_alive():
        print("\n\n[WAKTU HABIS] 2 menit berlalu. Sistem otomatis memproses SEMUA folder...")
        tes_input = ""  # Anggap kosong agar memproses semua
    else:
        if not tes_input:
            print("\n[INFO] Input dikosongkan. Memproses SEMUA folder...")
        else:
            print(f"\n[INFO] Memproses folder khusus: {tes_input}")

    target_numbers = set()
    if tes_input:
        for part in tes_input.split(','):
            part = part.strip()
            if '-' in part:
                try:
                    start, end = map(int, part.split('-'))
                    target_numbers.update(str(i) for i in range(start, end + 1))
                except ValueError:
                    print(f"[WARNING] Format rentang salah dan diabaikan: {part}")
            elif part:
                target_numbers.add(part)

    for folder in output_folders:
        if folder['mimeType'] != 'application/vnd.google-apps.folder':
            continue

        folder_name = folder['name']
        m = FOLDER_NUMBER_PATTERN.match(folder_name)
        if not m: continue

        number = m.group(1)
        if number not in updated_keperluan_map: continue

        # Eksekusi filter berdasarkan target_numbers
        if target_numbers and number not in target_numbers:
            continue

        excel_keperluan = updated_keperluan_map[number].upper()
        print(f"\n[COVER PROSES] Folder No. {number} -> Mengekstrak data riil berkas SPP-1...")

        sub_items = list_folder_items(folder['id'])
        spp1_file = next((f for f in sub_items if f['name'].lower() == "spp-1.pdf"), None)

        # Inisialisasi fallback diganti menjadi kosong/strip untuk memastikan real data
        var_anggaran = "-"
        var_spp_no = "-"
        var_pelaksana = "-"
        var_kode_apbg = "-"
        clean_spp_num = str(number)

        if spp1_file:
            try:
                spp1_bytes = download_file(spp1_file['id']).read()
                spp1_doc = fitz.open(stream=spp1_bytes, filetype="pdf")
                spp1_text = spp1_doc[0].get_text()

                match_no = re.search(r"NOMOR\s*:\s*([^\s\n]+)", spp1_text, re.IGNORECASE)
                if match_no:
                    var_spp_no = match_no.group(1).strip()
                    spp_num_match = re.search(r"(\d+)", var_spp_no)
                    if spp_num_match:
                        clean_spp_num = str(int(spp_num_match.group(1)))

                # --- PERBAIKAN EKSTRAKSI ANGGARAN (ANTI-KOSONG) ---
                match_rp = re.search(r"Jumlah Diminta\s*:\s*([\d.,]+)\s*Rp", spp1_text, re.IGNORECASE)
                if match_rp:
                    clean_rp = match_rp.group(1).replace('\n', '').replace(' ', '').strip()
                    clean_rp = re.sub(r'[,.]+$', '', clean_rp)
                    if any(char.isdigit() for char in clean_rp):
                        var_anggaran = f"RP {clean_rp}"

                match_pels = re.search(r"Pelaksana Kegiatan,\s*\n\s*([^\n]+)", spp1_text)
                if match_pels:
                    var_pelaksana = match_pels.group(1).strip().upper()

                spp1_doc.close()

                # --- KODE APBG diambil dari kolom Kode di tabel rincian SPP-2 (bukan SPP-1) ---
                spp2_file_for_kode = next((f for f in sub_items if f['name'].lower() == "spp-2.pdf"), None)
                if spp2_file_for_kode:
                    try:
                        spp2_bytes_kode = download_file(spp2_file_for_kode['id']).read()
                        spp2_doc_kode = fitz.open(stream=spp2_bytes_kode, filetype="pdf")
                        spp2_text_kode = spp2_doc_kode[0].get_text()
                        spp2_doc_kode.close()

                        # Kode APBG = kode rekening pada kolom "Kode" di tabel rincian SPP-2, format X.X.X.XX.
                        match_kode = re.search(r"(\d\.\d\.\d\.\d{2}\.)", spp2_text_kode)
                        if match_kode:
                            var_kode_apbg = match_kode.group(1).strip()
                    except Exception as ex_kode:
                        print(f"   • [WARNING] Gagal mengekstrak Kode APBG dari SPP-2: {ex_kode}")
                else:
                    print("   • [WARNING] Berkas SPP-2 tidak ditemukan untuk ekstraksi Kode APBG.")
                print(f"   -> [DATA DIKUNCI] No SPP: {var_spp_no} | Nilai: {var_anggaran} | Pelaksana: {var_pelaksana} | Kode: {var_kode_apbg}")
            except Exception as ex:
                print(f"   • [WARNING] Gagal mengekstrak teks SPP-1: {ex}")
        else:
            print("   • [WARNING] Berkas SPP-1 tidak ditemukan di folder ini. Variabel akan dikosongkan.")

        # 3. PROSES GENERASI COVER ELEGAN PREMIUM (TOTAL CLEAN LAYOUT)
        try:
            cover_buf = io.BytesIO()
            c = canvas.Canvas(cover_buf, pagesize=A4)
            width, height = A4
            center_x = width / 2.0

            # --- PALET WARNA EKSEKUTIF ---
            navy_blue = colors.Color(0.08, 0.18, 0.36)

            # --- FITUR SECURITY: WATERMARK GROUND PATTERN ---
            c.saveState()
            c.setFont("Helvetica-Bold", 55)
            c.setFillColor(colors.black)
            c.setFillAlpha(0.03)
            c.rotate(35)
            c.drawString(6*cm, 3*cm, "DOKUMEN RESMI")
            c.drawString(14*cm, -3*cm, "DOKUMEN RESMI")
            c.drawString(-1*cm, 10*cm, "DOKUMEN RESMI")
            c.restoreState()

            # --- HEADER: Kop Surat Pemerintahan ---
            c.setFillColor(navy_blue)
            c.setFont("Helvetica-Bold", 14)
            c.drawCentredString(center_x, height - 3.0*cm, "PEMERINTAH KABUPATEN ACEH BARAT")
            c.drawCentredString(center_x, height - 3.6*cm, "KECAMATAN SAMATIGA")
            c.drawCentredString(center_x, height - 4.2*cm, "GAMPONG COT MESJID")

            c.setFont("Helvetica-Bold", 12)
            c.drawCentredString(center_x, height - 5.0*cm, "TAHUN ANGGARAN 2026")

            # --- TITLE: Judul Utama Dokumen ---
            c.setFont("Helvetica-Bold", 32)
            c.drawCentredString(center_x, height - 8.0*cm, "LAPORAN")
            c.setFont("Helvetica-Bold", 24)
            c.drawCentredString(center_x, height - 9.0*cm, "PERTANGGUNGJAWABAN")

            # Sub-Deskripsi Dokumen
            c.setFillColor(colors.Color(0.3, 0.3, 0.3))
            c.setFont("Helvetica-Oblique", 10)
            c.drawCentredString(center_x, height - 9.8*cm, "Dokumen Resmi Rincian Pelaksanaan Kegiatan dan Administrasi Keuangan Gampong.")

            # --- KONFIGURASI GAYA TEKS AUTO-WRAP (ANTI-POTONG) ---
            styles = getSampleStyleSheet()
            style_value = ParagraphStyle(
                'ValueStyle',
                parent=styles['Normal'],
                fontName='Helvetica',
                fontSize=11,
                textColor=navy_blue,
                leading=14
            )
            style_value_bold = ParagraphStyle(
                'ValueStyleBold',
                parent=style_value,
                fontName='Helvetica-Bold'
            )
            style_pelaksana = ParagraphStyle(
                'PelaksanaStyle',
                parent=style_value,
                fontName='Helvetica-Bold',
                fontSize=16,
                leading=19
            )

            # --- LAYOUT FORMULIR DATA UTAMA ---
            data_form = [
                ["NAMA KEGIATAN", ":", Paragraph(excel_keperluan, style_value)],
                ["PELAKSANA KEGIATAN", ":", Paragraph(var_pelaksana, style_pelaksana)],
                ["SUMBER DANA", ":", Paragraph("ALOKASI DANA DESA (ADG)", style_value)],
                ["ANGGARAN", ":", Paragraph(var_anggaran, style_value_bold)],
                ["KODE APBG", ":", Paragraph(var_kode_apbg, style_value)],
                ["SPP NO", ":", Paragraph(var_spp_no, style_value)]
            ]

            table_layout = Table(data_form, colWidths=[5.5*cm, 0.5*cm, 10.0*cm])
            table_layout.setStyle(TableStyle([
                ('FONTNAME', (0,0), (0,-1), 'Helvetica-Bold'),
                ('FONTNAME', (1,0), (1,-1), 'Helvetica-Bold'),
                ('FONTSIZE', (0,0), (1,-1), 11),
                ('TEXTCOLOR', (0,0), (1,-1), navy_blue),
                ('VALIGN', (0,0), (-1,-1), 'TOP'),
                ('BOTTOMPADDING', (0,0), (-1,-1), 10),
                ('TOPPADDING', (0,0), (-1,-1), 4),
            ]))

            table_width, table_height = table_layout.wrapOn(c, width, height)
            pos_x = (width - table_width) / 2.0
            pos_y = height - 11.5*cm - table_height
            table_layout.drawOn(c, pos_x, pos_y)

            # --- FOOTER BLOCK KIRI & KANAN ---
            footer_base_y = 1.5*cm
            box_width = 4.5*cm
            box_height = 0.7*cm

            # 1. KIRI: Kotak Blok Nomor Arsip
            arsip_text = f"NO ARSIP: 2026{number.zfill(2)}"
            c.setFillColor(navy_blue)
            c.rect(3.5*cm, footer_base_y, box_width, box_height, fill=1, stroke=0)

            c.setFillColor(colors.white)
            c.setFont("Helvetica-Bold", 10)
            c.drawCentredString(3.5*cm + (box_width / 2.0), footer_base_y + 0.22*cm, arsip_text)

            # 2. KANAN: Kotak QR Kompak
            qr_size = 2.90 * cm
            margin = 0.01 * cm
            label_h = 0.65 * cm

            qr_total_w = qr_size + (margin * 2)
            qr_total_h = qr_size + label_h + (margin * 2)

            qr_box_x = width - 3.5 * cm - qr_total_w

            safe_w = qr_total_w - 0.04 * cm
            safe_h = qr_total_h - 0.04 * cm
            offset = 0.02 * cm

            qr_drawing = Drawing(qr_total_w, qr_total_h)

            qr_drawing.add(Rect(offset, offset, safe_w, safe_h, fillColor=colors.white, strokeColor=colors.black, strokeWidth=1.5))
            qr_drawing.add(Rect(offset, offset, safe_w, label_h, fillColor=colors.black, strokeColor=colors.black))

            text_element = String(qr_total_w / 2.0, 0.22*cm, "AKSES DIGITAL", textAnchor='middle')
            text_element.fontName = "Helvetica-Bold"
            text_element.fontSize = 8
            text_element.fillColor = colors.white
            qr_drawing.add(text_element)

            qr_url = f"https://cotmesjid.github.io/arsip/2/{clean_spp_num}"
            qr_code = qr.QrCodeWidget(qr_url)
            qr_code.barWidth = qr_size
            qr_code.barHeight = qr_size
            qr_code.qrVersion = 3

            qr_code.x = margin
            qr_code.y = label_h + margin

            qr_drawing.add(qr_code)
            qr_drawing.drawOn(c, qr_box_x, footer_base_y)

            c.save()
            final_cover_bytes = cover_buf.getvalue()

            # 4. MODIFIKASI NAMA FILE: SPP-{Nomor}-COVER.pdf
            cover_filename = f"SPP-{clean_spp_num}-COVER.pdf"

            # Hapus berkas lama jika ada (mengecek nama lama atau pola "-COVER.pdf")
            old_cover = next((f for f in sub_items if f['name'] == cover_filename or f['name'] == "00 - Cover LPJ Premium.pdf" or f['name'].endswith("-COVER.pdf")), None)
            if old_cover:
                try: drive_service.files().delete(fileId=old_cover['id'], supportsAllDrives=True).execute()
                except: pass

            upload_file(cover_filename, final_cover_bytes, folder['id'])
            print(f"   -> [SUKSES PRESTISIUS] Berkas '{cover_filename}' berhasil diperbarui.")
            total_cover_created += 1

            # 5. PROSES PENGGABUNGAN PDF SECARA AMAN (ANTI-404)
            updated_sub_items = list_folder_items(folder['id'])
            old_merge = next((f for f in updated_sub_items if f['name'] == GABUNG_NAME), None)
            if old_merge:
                try: drive_service.files().delete(fileId=old_merge['id'], supportsAllDrives=True).execute()
                except: pass

            gabung_order = [final_cover_bytes]

            def safe_download(file_obj):
                try:
                    return download_file(file_obj['id']).read()
                except Exception as dl_err:
                    print(f"   • [SKIP] Berkas {file_obj['name']} dilewati karena tidak ditemukan (404).")
                    return None

            spp1_file_obj = next((f for f in updated_sub_items if f['name'].lower() == "spp-1.pdf"), None)
            if spp1_file_obj:
                spp1_b = safe_download(spp1_file_obj)
                if spp1_b: gabung_order.append(spp1_b)

            spp2_file_obj = next((f for f in updated_sub_items if f['name'].lower() == "spp-2.pdf"), None)
            if spp2_file_obj:
                spp2_b = safe_download(spp2_file_obj)
                if spp2_b: gabung_order.append(spp2_b)

            sptb_file_obj = next((f for f in updated_sub_items if f['name'].lower() == "sptb.pdf"), None)
            if sptb_file_obj:
                sptb_b = safe_download(sptb_file_obj)
                if sptb_b: gabung_order.append(sptb_b)

            kwt_items = sorted([f for f in updated_sub_items if KWT_PATTERN.match(f['name'])], key=lambda f: f['name'])
            for f in kwt_items:
                kwt_b = safe_download(f)
                if kwt_b: gabung_order.append(kwt_b)

            if len(gabung_order) > 1:
                gabung_bytes = merge_pdfs(gabung_order)
                upload_file(GABUNG_NAME, gabung_bytes, folder['id'])
                print(f"   -> [BUNDEL SUKSES] Berkas gabungan '{GABUNG_NAME}' sukses disinkronisasi.")

        except Exception as e:
            print(f"   -> [ERROR] Gagal memproses folder nomor {number}: {e}")

else:
    print("[STOP] Tidak ada data di 'updated_keperluan_map'. Proses dihentikan.")

print(f"\n[SELESAI] Sukses! Seluruh sistem cover eksekutif clean anti-pemalsuan selesai dikonfigurasi.")